In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
from tqdm import tqdm
import numpy as np
import pandas as pd
import nltk
import matplotlib.pyplot as plt
import torch
import math
import torchvision
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import matplotlib.pyplot as plt
import time
import os
import copy

import os
import re
from tqdm import tqdm
import numpy as np
import pandas as pd
import nltk
import matplotlib.pyplot as plt
import torch
import csv

In [3]:
def load_from_CSV(path,category):
  with open(path) as csvfile:
    reader = csv.DictReader(csvfile)
    texts = []
    labels2 = []
    for row in reader:
      #print(row['Comment'])
      #print(row['Tag'])
      #break
      comments = str(row['Comment'])
      texts.append(comments)
      labels2.append(category)
      #texts.append(line.decode(errors='ignore').lower().strip())
     # labels2.append(category)

  return texts, labels2
Bully_text, Bully_labels = load_from_CSV('/content/drive/MyDrive/CyberBully/Bully.csv', category=0)
Non_Bully_text, Non_Bully_labels = load_from_CSV('/content/drive/MyDrive/CyberBully/NotBully.csv', category=1)
texts = np.array(Bully_text + Non_Bully_text)
labels = np.array(Bully_labels+Non_Bully_labels)
labels = np.array([0]*len(Bully_text) + [1]*len(Non_Bully_text) )

In [4]:
def preprocess(text):
    text = str(text).replace('।', '\n')
    whitespace = re.compile(u"[\s\u0020\u00a0\u1680\u180e\u202f\u205f\u3000\u2000-\u200a]+", re.UNICODE)
    bangla_fullstop = u"\u0964"
    punctSeq = u"['\"“”‘’]+|[.?!,…]+|[:;]+"
    punc = u"[(),$%^&*+={}\[\]:\"|\'\~`<>/,¦!?½£¶¼©⅐⅑⅒⅓⅔⅕⅖⅗⅘⅙⅚⅛⅜⅝⅞⅟↉¤¿º;-]+"
    text = whitespace.sub(" ", text).strip()
    text = re.sub(punctSeq, " ", text)
    text = re.sub(punc, " ", text)
    text = "".join(i for i in text if ord(i) > ord('z') or ord(i) == 32)
    text = re.sub(' +', ' ', text)
    return (text)


In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f'There are {torch.cuda.device_count()} GPU(s) available.')
    print('Device name:', torch.cuda.get_device_name(0))

else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

There are 1 GPU(s) available.
Device name: Tesla T4


In [6]:
from nltk.tokenize import word_tokenize
from collections import defaultdict

def tokenize(texts):
  max_len = 0
  tokenized_texts = []
  word2idx ={}

  word2idx['<pad>'] = 0
  word2idx['<unk>'] = 1
  idx = 2
  for sent in texts:
    tokenized_sent = sent.split()
    tokenized_texts.append(tokenized_sent)
    for token in tokenized_sent:
      if token not in word2idx:
        word2idx[token] = idx
        idx += 1
    max_len = max(max_len, len(tokenized_sent))
  return tokenized_texts, word2idx, max_len

def encode(tokenized_texts, word2idx, max_len):
  input_ids = []
  for tokenized_sent in tokenized_texts:
    tokenized_sent += ['<pad>'] * (max_len - len(tokenized_sent))
    input_id = [word2idx.get(token) for token in tokenized_sent]
    input_ids.append(input_id)
  return np.array(input_ids)

In [7]:

def load_glove(word2idx, filenameg, vector_size):
  embedding_vectors = np.random.uniform(-0.25, 0.25, (len(word2idx), vector_size))
  f = open(filenameg,encoding='utf-8', errors='ignore')
  embedding_vectors[word2idx['<pad>']] = np.zeros((vector_size,))
  count = 0
  for line in f:
    values = line.split()
    word = values[0]
    #print(word)
    #print(values[1:])
    vectorg = np.asarray(values[1:], dtype="float32")
    if word in word2idx:
      count += 1
      embedding_vectors[word2idx[word]] = vectorg
      #print(f"There are {count} / {len(word2idx)} pretrained vectors found.")
  f.close()
  print(count)
  return embedding_vectors

In [8]:
# Tokenize, build vocabulary, encode tokens
print("Tokenizing...\n")
tokenized_texts, word2idx, max_len = tokenize(texts)
input_ids = encode(tokenized_texts, word2idx, max_len)

print(len(word2idx))
embedding_dim = 100
# Load pretrained vectors
embeddings = load_glove(word2idx, '/content/drive/MyDrive/CyberBully/GloVe100d.txt',100)
embeddings = torch.tensor(embeddings)

Tokenizing...

98477
52330


In [9]:
from torch.utils.data import (TensorDataset, DataLoader, RandomSampler,SequentialSampler)

def data_loader(train_inputs, val_inputs, train_labels, val_labels,batch_size=20):
  train_inputs, val_inputs, train_labels, val_labels =\
  tuple(torch.tensor(data) for data in[train_inputs, val_inputs, train_labels, val_labels])
  batch_size = 20
  train_data = TensorDataset(train_inputs, train_labels)
  train_sampler = RandomSampler(train_data)
  train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
  val_data = TensorDataset(val_inputs, val_labels)
  val_sampler = SequentialSampler(val_data)
  val_dataloader = DataLoader(val_data, sampler=val_sampler, batch_size=batch_size)
  return train_dataloader, val_dataloader

In [10]:
from sklearn.model_selection import train_test_split

train_inputs, val_inputs, train_labels, val_labels = train_test_split(input_ids, labels, test_size=0.3, random_state=42)

train_dataloader, val_dataloader =data_loader(train_inputs, val_inputs, train_labels, val_labels, batch_size=20)

In [11]:
class CNN_NLP(nn.Module):
  def __init__(self):
    super(CNN_NLP, self).__init__()
    pretrained_embedding=embeddings
    freeze_embedding=False
    vocab_size=len(word2idx)
    embed_dim=100
    filter_sizes=[3, 4, 5]
    num_filters=[100, 100, 100]
    num_classes=3
    dropout=0.5
    self.vocab_size, self.embed_dim = pretrained_embedding.shape
    self.embedding = nn.Embedding.from_pretrained(pretrained_embedding,freeze=freeze_embedding)
    self.conv1 =  nn.Conv1d(in_channels=self.embed_dim,out_channels=100,kernel_size=3)
    self.conv2 =  nn.Conv1d(in_channels=self.embed_dim,out_channels=100,kernel_size=4)
    self.conv3 =  nn.Conv1d(in_channels=self.embed_dim,out_channels=100,kernel_size=5)
    self.fc = nn.Linear(np.sum(num_filters), num_classes)
    self.dropout = nn.Dropout(p=dropout)
  def forward(self, input_ids):
    x_embed = self.embedding(input_ids).float()
    x_reshaped = x_embed.permute(0, 2, 1)
    x1 = F.relu(self.conv1(x_reshaped))
    x2 = F.relu(self.conv2(x_reshaped))
    x3 = F.relu(self.conv3(x_reshaped))
    x1 = F.max_pool1d(x1,kernel_size=x1.shape[2])
    x2 = F.max_pool1d(x2,kernel_size=x2.shape[2])
    x3 = F.max_pool1d(x3,kernel_size=x3.shape[2])
    fc_x = torch.cat([x1.squeeze(dim=2),x2.squeeze(dim=2),x3.squeeze(dim=2)], dim=1)
    logits = self.fc(self.dropout(fc_x))
    return logits

In [12]:
import torch.optim as optim

def initilize_model():
  cnn_model = CNN_NLP()
  cnn_model.to(device)
  optimizer = optim.Adadelta(cnn_model.parameters(),lr=0.25,rho=0.95)
  return cnn_model, optimizer

In [13]:
import random
import time

# Specify loss function
loss_fn = nn.CrossEntropyLoss()

def set_seed(seed_value=42):
    """Set seed for reproducibility."""

    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)

def train(model, optimizer, train_dataloader, val_dataloader=None, epochs=15):
  best_accuracy = 0
  print("Start training...\n")
  print(f"{'Epoch':^7} | {'Train Loss':^12} | {'Val Loss':^10} | {'Val Acc':^9} | {'Elapsed':^9}")
  print("-"*60)
  best_model = copy.deepcopy( model.state_dict() )
  for epoch_i in range(epochs):
    t0_epoch = time.time()
    total_loss = 0
    model.train()
    for step, batch in enumerate(train_dataloader):
      b_input_ids, b_labels = tuple(t.to(device) for t in batch)
      model.zero_grad()
      logits = model(b_input_ids)
      #print('logits',logits)
      #print('b_labels',b_labels)
      loss = loss_fn(logits, b_labels)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      avg_train_loss = total_loss / len(train_dataloader)
    #print('Loss: ',total_loss)

    model.eval()
    val_accuracy = []
    val_loss = []
    for batch in val_dataloader:
      b_input_ids, b_labels = tuple(t.to(device) for t in batch)
      with torch.no_grad():
        logits = model(b_input_ids)
        loss = loss_fn(logits, b_labels)
        val_loss.append(loss.item())
        preds = torch.argmax(logits, dim=1).flatten()
      #print(preds)
        accuracy = (preds == b_labels).cpu().numpy().mean() * 100
        val_accuracy.append(accuracy)
    val_loss = np.mean(val_loss)
    val_accuracy = np.mean(val_accuracy)
    time_elapsed = time.time() - t0_epoch
    print(f"{epoch_i + 1:^7} | {avg_train_loss:^12.6f} | {val_loss:^10.6f} | {val_accuracy:^9.2f} | {time_elapsed:^9.2f}")
    #print('val_accuracy: ',val_accuracy)
    if val_accuracy > best_accuracy:
      best_accuracy = val_accuracy
      best_model = copy.deepcopy( model.state_dict() )



  return best_model

def evaluate(model, val_dataloader):
  model.eval()
  val_accuracy = []
  val_loss = []
  y_act = []
  y_predict = []
  for batch in val_dataloader:
    b_input_ids, b_labels = tuple(t.to(device) for t in batch)
    with torch.no_grad():
      logits = model(b_input_ids)
      loss = loss_fn(logits, b_labels)
      val_loss.append(loss.item())
      preds = torch.argmax(logits, dim=1).flatten()
      y_predict.append(preds)
      y_act.append(b_labels)
      #print(preds)
      accuracy = (preds == b_labels).cpu().numpy().mean() * 100
      val_accuracy.append(accuracy)
  val_loss = np.mean(val_loss)
  val_accuracy = np.mean(val_accuracy)
  #print('actual', y_act)
  #print('actual', y_predict)
  return val_loss, val_accuracy, y_act, y_predict

In [14]:
set_seed(42)
cnn_non_static, optimizer = initilize_model()
best_model = train(cnn_non_static, optimizer, train_dataloader, val_dataloader, epochs=10)

Start training...

 Epoch  |  Train Loss  |  Val Loss  |  Val Acc  |  Elapsed 
------------------------------------------------------------
   1    |   0.392665   |  0.340980  |   84.51   |   30.49  
   2    |   0.341700   |  0.326606  |   85.11   |   27.83  
   3    |   0.319711   |  0.317665  |   85.71   |   28.16  
   4    |   0.299836   |  0.309998  |   86.09   |   28.05  
   5    |   0.277905   |  0.300577  |   86.84   |   28.28  
   6    |   0.256923   |  0.298187  |   86.98   |   28.23  
   7    |   0.241753   |  0.300523  |   87.16   |   28.34  
   8    |   0.220828   |  0.301483  |   87.18   |   29.99  
   9    |   0.201202   |  0.301428  |   87.74   |   28.68  
  10    |   0.180974   |  0.306478  |   87.28   |   28.69  


In [15]:
torch.save(best_model,'/content/drive/MyDrive/CyberBully/CNN-Bully.pt')

In [16]:
cnn_model = CNN_NLP()
cnn_model.to(device)
print(val_dataloader.batch_sampler)
cnn_model.load_state_dict(torch.load('/content/drive/MyDrive/CyberBully/CNN-Bully.pt'))
val_loss, val_accuracy, y_act, y_predict = evaluate(cnn_model,val_dataloader)
print("Final_model_Accuracy", val_accuracy)

<ipython-input-16-8010f45ee5c4>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load('/content/drive/MyDrive/CyberBully/CNN-Bully.pt'))


Final_model_Accuracy 87.73635153129162


In [17]:
from sklearn import metrics
y = [t.cpu().numpy() for t in y_act]
yp = [t.cpu().numpy() for t in y_predict]
y = [item for sublist in y for item in sublist]
yp = [item for sublist in yp for item in sublist]
print(metrics.classification_report(y, yp))
print(metrics.confusion_matrix(y, yp))

              precision    recall  f1-score   support

           0       0.91      0.91      0.91     10080
           1       0.82      0.81      0.81      4924

    accuracy                           0.88     15004
   macro avg       0.86      0.86      0.86     15004
weighted avg       0.88      0.88      0.88     15004

[[9202  878]
 [ 960 3964]]
